# 04. Analisis Ketepatan Sasaran Bansos & Welfare Priority Ranking
## Audit Inclusion Error, Exclusion Error & Rekomendasi Alokasi Bantuan

Notebook ini memuat tahapan **Evaluation & Deployment**:
1. Profiling karakteristik setiap klaster dan auto-labeling.
2. Perhitungan **Inclusion Error** dan **Exclusion Error** alokasi bansos.
3. Penyusunan **Welfare Priority Index (WPI)** untuk menentukan daftar prioritas intervensi bantuan per daerah se-Indonesia.
4. Pelatihan model klasifikasi terawasi (**Random Forest & XGBoost**) untuk memprediksi kelayakan intervensi bantuan sosial.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.targeting import (
    assign_ordered_cluster_labels, compute_cluster_profiles,
    analyze_targeting_errors, compute_welfare_priority_ranking
)
from src.classification import WelfareClassifierEngine

# Load Hasil Pipeline
df = pd.read_csv('../data/processed/hasil_audit_bansos_wilayah.csv')
print(f'Total data hasil audit: {len(df)} Kabupaten/Kota')
df.head()

In [ ]:
# 1. Profil Karakteristik Tiap Klaster
profiles = compute_cluster_profiles(df)
profiles

In [ ]:
# 2. Analisis Ketepatan Sasaran Bansos (Inclusion & Exclusion Error)
error_stats = analyze_targeting_errors(df)
print('=== HASIL AUDIT SALAH SASARAN BANSOS NASIONAL ===')
print(f'Exclusion Error Wilayah : {error_stats["exclusion_error_wilayah_pct"]}%')
print(f'Inclusion Error Wilayah : {error_stats["inclusion_error_wilayah_pct"]}%')
print('\nBreakdown Error per Pulau:')
error_stats['error_by_island_df']

In [ ]:
# 3. Top 15 Daerah Paling Prioritas Berdasarkan Welfare Priority Index (WPI)
cols_show = ['ranking_prioritas_bansos', 'kabupaten_kota', 'provinsi', 'klaster_label', 
             'welfare_priority_index_wpi', 'persentase_penduduk_miskin', 'rasio_cakupan_bansos', 'rekomendasi_kebijakan']
df[cols_show].head(15)

In [ ]:
# 4. Evaluasi Model Klasifikasi Kelayakan Bansos (Random Forest & XGBoost)
clustering_features = [
    'persentase_penduduk_miskin', 'garis_kemiskinan', 'indeks_kedalaman_p1', 'indeks_keparahan_p2',
    'pengeluaran_per_kapita', 'indeks_pembangunan_manusia_ipm', 'persentase_sanitasi_layak',
    'persentase_air_minum_layak', 'persentase_rumah_layak_huni', 'indeks_infrastruktur_dasar',
    'indeks_kemiskinan_multidimensi_ikm'
]

clf = WelfareClassifierEngine(random_state=42)
X_train, X_test, y_train, y_test = clf.prepare_data(df, feature_cols=clustering_features)
clf.train_models(X_train, y_train)
eval_dict = clf.evaluate_models(X_test, y_test)

print('F1-Score RF     :', eval_dict['Random Forest']['f1_score'])
print('F1-Score XGBoost:', eval_dict['XGBoost']['f1_score'])

# Feature Importance Ranking
imp_df = clf.get_feature_importance()
plt.figure(figsize=(10, 6))
sns.barplot(data=imp_df, x='Rata-Rata Importance', y='Fitur', palette='viridis')
plt.title('Feature Importance Indikator BPS terhadap Kelayakan Prioritas Bansos', fontsize=12, fontweight='bold')
plt.show()
imp_df